In [ ]:
from pyskills import doc
from pyskills import *

In [ ]:
# from pote.server import stop_server_a

In [ ]:
# await stop_server_a()

What does `pote.server` do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`pote.server` manages FastHTML/Uvicorn web servers from a notebook or Solveit session.

Its role is to make an interactive app server easy to work with while you develop:

- **start or reuse** a server instead of accidentally starting duplicates
- **inspect** the running app/server state
- **preview** the app in the browser
- **clean up** a server when it is no longer needed
- **dynamically load JavaScript modules** into the active browser session

For example, the loaded setup uses `ensure_server_a(...)` to obtain or reuse a server and its associated FastHTML app, route helper, and related state.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 39201, "completion_tokens": 311, "total_tokens": 39512, "cached_tokens": 3456, "reasoning_tokens": 170, "cost": 0.006113400000000001}
```


Give me simple examples demoing their features.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I’ll inspect the module’s current API first, so the examples use the actual function names and signatures.
- ⏳ `py(code="import pote.server; doc(pote.server)")` ⏳


````json {.tool}
{
  "id": "fc_01dd4459005487f3016a7e06d3781c8195bcac7479de45c82e",
  "name": "py",
  "args": {
    "code": "import pote.server; doc(pote.server)"
  },
  "result": "<TRUNCATED>…ote.server:\n\n\"\"\"Start, reuse, inspect, preview, and clean up FastHTML/Uvicorn servers in Jupyter or Solveit, and dynamically import JavaScript modules in the active browser.\n\n### Choosing a helper\n\n```python\nfrom pote.server import ensure_server_a, serve_app, setup_mod, add_js_msg\n```\n\n- For a persistent interactive app, use `await ensure_server_a(...)`.\n- For a scoped server with automatic cleanup, including tests, use `async with serve_app(...)`.\n- To inspect or stop notebook server state, use `find_server`, `is_serving_a`, and `stop_server_a`.\n- To preview a FastHTML route with an iframe, use the returned `p` helper.\n- For iterative browser JavaScript, use `setup_mod(..., versioned=True)`.\n- To extract JavaScript from a `%%js` or `%%javascript` message, use `get_js`, `add_js_msg`, or `setup_mod` without `src`.\n\nThe async APIs are normally preferable in Solveit and Jupyter. These helpers target FastHTML applications; the underlying `JupyUvi` can serve more general ASGI applications, but `pote.server` does not promise that compatibility.\n\n### Persistent server\n\n```python\nsrv,app,rt,p = await ensure_server_a(port=None)\n\n@rt\ndef hello(): return 'hello'\n\np('/hello')\nawait stop_server_a(srv.port)\n```\n\n`ensure_server_a` returns persistent notebook state as `(srv, app, rt, p)`:\n\n- `srv` controls the uvicorn lifecycle.\n- `app` is the application actually being served.\n- `rt` is its FastHTML route decorator.\n- `p` previews routes through the returned server.\n\nSelection precedence is:\n\n1. An explicit live `srv`.\n2. A live `JupyUvi` already bound to the requested port.\n3. A newly created server.\n\nWhen an explicit or discovered server is reused, `app`, `app_kw`, and `jupyuvi_kw` are ignored. For an explicit `srv`, its own port is authoritative and the requested `port` is ignored. `mods` are still registered, and the ESM route is installed on the reused app. `port=None` selects a free port outside Solveit’s public-port allocation.\n\nThe server remains live until explicitly stopped. In notebooks, prefer `await stop_server_a(srv.port)`, which supports synchronous and asynchronous servers and clears all namespace aliases to the stopped server.\n\n### Temporary server and testing\n\nUse `serve_app` when a server should exist only for a block:\n\n```python\nasync with serve_app(port=None) as srv:\n    @srv.app.get('/hello')\n    def hello(): return 'hello'\n\n    cli = TestClient(srv.app)\n    test_eq(cli.get('/hello').text, 'hello')\n```\n\nA server created by the context is owned by it: cleanup runs after normal exit, setup or body errors, and cancellation. By default, routes added to an owned app during the block are removed on exit.\n\nA reused server is not owned by the context. It remains live afterward, and route changes made to it persist.\n\nThe context yields only the live server; use `srv.app`, `srv.port`, and other server attributes inside the block. In tests, `port=None` avoids persistent and publicly allocated ports.\n\n`TestClient(srv.app)` tests the ASGI application directly and does not prove that uvicorn is reachable over the network. Use `await is_serving_a(srv)` when network availability is part of the contract.\n\n### Browser ES modules\n\nRegister and import JavaScript in the active Solveit browser with:\n\n```python\nsrc = 'export const answer = 42'\nnm,res = await setup_mod('editor', src, versioned=True)\n```\n\nWith `versioned=True`, repeated calls register successive names such as `editor_1` and `editor_2`, and import distinct URLs such as `/esm/editor?v=1`. Use this during iterative development to bypass the browser’s ES-module cache.\n\nThe module registry belongs to the current Python kernel, not to an app or server. Registering an existing key replaces its source, and registered entries survive server shutdown. There is currently no public operation for unregistering an entry or clearing the registry.\n\nRegistered modules are executable browser code and may be served through the app’s public HTTP endpoint. Register only trusted source; module names are not an authorization boundary.\n\nAn unversioned name provides a stable URL. Replacing its registered source changes future HTTP responses, but a browser that has already imported that URL may continue using its cached module.\n\n`setup_mod` and `load_mod` always import through the server on port 8000 in the active Solveit browser. They do not use the most recently created server, a server returned on another port, or an arbitrary explicit server. If no port-8000 server exists, one is created for the import and stopped afterward; the imported browser module remains loaded and the registered source remains in the kernel registry.\n\nJavaScript can also come from a dialog message whose complete first line is `%%js` or `%%javascript`:\n\n- `get_js` returns the source without the magic line.\n- `setup_mod` without `src` extracts and imports the preceding JavaScript message.\n- `add_js_msg` creates linked Python state containing the extracted source.\n\nA missing registry entry…</TRUNCATED>"
}
````





Here are small, independent examples of the main features.

**Start or reuse a persistent server**

```python
from pote.server import ensure_server_a, stop_server_a
srv, app, rt, p = await ensure_server_a(port=None)
```

`ensure_server_a` either reuses a live compatible server or creates one. `port=None` chooses a free non-public port.

**Add and preview a route**

```python
@rt
def hello(): return "Hello from FastHTML"

p("/hello")
```

`p` displays that route in an iframe. The server remains active until stopped:

```python
await stop_server_a(srv.port)
```

**Check whether a server is reachable**

```python
from pote.server import is_serving_a
await is_serving_a(srv)
```

This returns `True` only when the server appears started and responds successfully to `GET /`.

**Find an existing server**

```python
from pote.server import find_server
find_server(port=8000)
```

This returns the server object and the namespace name bound to it, or `(None, None)` if no matching live server is known.

**Use a temporary server for a test**

```python
from pote.server import serve_app

async with serve_app(port=None) as srv:
    @srv.app.get("/hello")
    def hello(): return "hello"
```

A server created for the block is stopped automatically afterward. This is useful for tests and short experiments.

**Register and load browser JavaScript**

```python
from pote.server import setup_mod
nm, result = await setup_mod("maths", "export const answer = 42")
```

This registers the module as `maths` and imports it into the active Solveit browser. For iterative edits that bypass browser module caching:

```python
nm, result = await setup_mod("maths", "export const answer = 43", versioned=True)
```

That creates a versioned name such as `maths_1` and loads it from a distinct URL.

**Register JavaScript without loading it yet**

```python
from pote.server import reg_mod
reg_mod("helpers", "export const square = x => x * x")
```

The source stays registered for this Python kernel and can later be loaded with:

```python
from pote.server import load_mod
await load_mod("helpers")
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 81464, "completion_tokens": 986, "total_tokens": 82450, "cached_tokens": 41728, "reasoning_tokens": 434, "cost": 0.012959100000000001}
```


**Start or reuse a persistent server**

In [ ]:
from pote.server import ensure_server_a, stop_server_a
srv, app, rt, p = await ensure_server_a()

`ensure_server_a` either reuses a live compatible server or creates one.

**Add and preview a route**

In [ ]:
@rt
def hello(): return "Hello from FastHTML"

p("/hello")

HTML(<iframe src="http://localhost:8000/hello" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

`p` displays that route in an iframe. The server remains active until stopped:

In [ ]:
await stop_server_a(srv.port)

**Check whether a server is reachable**

In [ ]:
from pote.server import is_serving_a
await is_serving_a(srv)

This returns `True` only when the server appears started and responds successfully to `GET /`.

**Find an existing server**

In [ ]:
from pote.server import find_server
find_server(port=8000)

This returns the server object and the namespace name bound to it, or `(None, None)` if no matching live server is known.

**Use a temporary server for a test**

In [ ]:
from pote.server import serve_app

async with serve_app(port=None) as srv:
    @srv.app.get("/hello")
    def hello(): return "hello"

A server created for the block is stopped automatically afterward. This is useful for tests and short experiments.

**Register and load browser JavaScript**

In [ ]:
from pote.server import setup_mod
nm, result = await setup_mod("maths", "export const answer = 42")

This registers the module as `maths` and imports it into the active Solveit browser. For iterative edits that bypass browser module caching:

In [ ]:
nm, result = await setup_mod("maths", "export const answer = 43", versioned=True)

That creates a versioned name such as `maths_1` and loads it from a distinct URL.

**Register JavaScript without loading it yet**

In [ ]:
from pote.server import reg_mod
reg_mod("helpers", "export const square = x => x * x")

The source stays registered for this Python kernel and can later be loaded with:

In [ ]:
from pote.server import load_mod
await load_mod("helpers")